In [ ]:
# pip install mlflow
# mlflow server --host 127.0.0.1 --port 8080   



In [1]:
import mlflow
from mlflow.models import infer_signature 

In [2]:
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [3]:
# Load the Iris dataset
X, y = datasets.load_iris(return_X_y=True)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [4]:
clf = DecisionTreeClassifier(max_depth=4, criterion='gini')
clf.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=4)

In [5]:
# Define the model hyperparameters
params = {
    "solver": "lbfgs",
    "max_iter": 1000,
    "multi_class": "multinomial",
    "random_state": 50,
}

In [6]:
# Train the model
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

c:\Users\murzi\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression(max_iter=1000, multi_class='multinomial', random_state=50)

In [7]:
# Predict on the test set
y_pred_clf = clf.predict(X_test)
y_pred_lr = lr.predict(X_test)

In [8]:
# Calculate metrics
accuracy_clf = accuracy_score(y_test, y_pred_clf)
accuracy_lr = accuracy_score(y_test, y_pred_lr)
accuracy_clf, accuracy_lr

(1.0, 1.0)

In [9]:
# Set our tracking server uri for logging
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

# Create a new MLflow Experiment
mlflow.set_experiment("MLflow Quickstart test") 

2025/05/03 21:47:07 INFO mlflow.tracking.fluent: Experiment with name 'MLflow Quickstart test' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/111813585103607288', creation_time=1746283627950, experiment_id='111813585103607288', last_update_time=1746283627950, lifecycle_stage='active', name='MLflow Quickstart test', tags={}>

In [10]:
# Start an MLflow run
with mlflow.start_run():
    # Log the hyperparameters
    mlflow.log_params(params)

    # Log the loss metric
    mlflow.log_metric("LogisticRegression accuracy", accuracy_lr)
    mlflow.log_metric("DecisionTreeClassifier accuracy", accuracy_clf)

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training Info", "Basic LR model for iris data")

    # Infer the model signature
    signature = infer_signature(X_train, lr.predict(X_train))

    # Log the model
    model_info = mlflow.sklearn.log_model(
        sk_model=lr,
        artifact_path="iris_model",
        signature=signature,
        input_example=X_train,
        registered_model_name="tracking-quickstart",
    )

Successfully registered model 'tracking-quickstart'.
2025/05/03 21:47:21 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tracking-quickstart, version 1
Created version '1' of model 'tracking-quickstart'.


🏃 View run smiling-gull-579 at: http://127.0.0.1:8080/#/experiments/111813585103607288/runs/bec35cf0e2f94fa1b886286eb2a1b2f9
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/111813585103607288


А как загрузить модель?

После логгирования модели мы можем:

- Загрузив модель с использованием pyfunc от MLflow.
- Запустить Predict для новых данных с использованием загруженной модели.

In [11]:
sklearn_pyfunc = mlflow.pyfunc.load_model(model_uri=model_info.model_uri) 

In [12]:
pred_load = sklearn_pyfunc.predict(X_test)
pred_load

array([1, 0, 2, 1, 1, 0, 1, 2, 1, 1, 2, 0, 0, 0, 0, 1, 2, 1, 1, 2, 0, 2,
       0, 2, 2, 2, 2, 2, 0, 0])

Данные для обучения ирисов, которые мы использовали, представляют собой структуру массива NumPy.
Однако мы также можем передать метод predict фрейм данных Pandas, как показано ниже.

In [ ]:
iris_feature_names = datasets.load_iris().feature_names

result = pd.DataFrame(X_test, columns=iris_feature_names)
result["actual_class"] = y_test
result["predicted_class"] = pred_load

result[:4]

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),actual_class,predicted_class
0,6.1,2.8,4.7,1.2,1,1
1,5.7,3.8,1.7,0.3,0,0
2,7.7,2.6,6.9,2.3,2,2
3,6.0,2.9,4.5,1.5,1,1
